# 01 | Market Data Engineering and Currency Normalization

**Project Alpha 2027 — Quantitative Financial Research**  
**Status:** research notebook; no trading execution or broker connectivity.

## Research question
How can a panel of European equities quoted in different currencies be converted into an auditable, EUR-denominated, aligned historical dataset?

## Input contracts
- `data/prices_real_2026-11-04.csv`: daily adjusted price panel in **local quotation currencies**; columns are ticker IDs, index is the date. This file is private and **not included**.
- `data/eur_nok_aligned_full.csv`: a **daily** aligned EUR/NOK series with one column named `NOK_per_EUR`, covering **every date** in the price panel. Obtain or construct it independently from historical FX data. The small three-day FX audit is **not** sufficient to convert the full panel.
- Five identified Oslo-listed securities are converted by `local_NOK / NOK_per_EUR`.

The cleaned EUR panel from the user's historical Colab work was previously saved as `prices_eur_2026-11-04.csv`. This notebook provides a **new, transparent reconstruction path** and validates against that reference when present. No new prices are fabricated.

**Methodological cautions.** Yahoo-adjusted prices may incorporate dividends and corporate actions, while the Trading Game may mark positions using different price conventions. For historical FX returns, convention must be dated and consistent; nominal stock prices and adjusted prices should not be mixed. A downloaded FX quote is not the NextWise transaction conversion rate.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# Run from the repository root, or from within notebooks/.
ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
DATA = ROOT / "data"
OUTPUT = ROOT / "reports" / "research_outputs"
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Repository root:", ROOT.resolve())
print("Data directory:", DATA.resolve())

## 1. Declare the currency mapping

The mapping below is derived from the historical notebook. Do not assume other European listings are in EUR without an independent check of exchange and instrument currency.

In [ ]:
OSLO_NOK = {"EQNR.OL", "AKRBP.OL", "KOG.OL", "YAR.OL", "TEL.OL"}
RAW = DATA / "prices_real_2026-11-04.csv"
FX = DATA / "eur_nok_aligned_full.csv"
if not RAW.is_file() or not FX.is_file():
    raise FileNotFoundError(
        "Required: data/prices_real_2026-11-04.csv and "
        "data/eur_nok_aligned_full.csv. The complete FX series is not supplied "
        "with this GitHub package."
    )
local = pd.read_csv(RAW, index_col=0, parse_dates=True)
local.index = pd.to_datetime(local.index).normalize()
local = local.apply(pd.to_numeric, errors="coerce").sort_index()
fx = pd.read_csv(FX, index_col=0, parse_dates=True)
fx.index = pd.to_datetime(fx.index).normalize()
if "NOK_per_EUR" not in fx.columns:
    raise ValueError("FX file requires column: NOK_per_EUR")
fx = pd.to_numeric(fx["NOK_per_EUR"], errors="coerce")
if not OSLO_NOK.issubset(local.columns):
    raise ValueError(f"Missing NOK tickers: {sorted(OSLO_NOK-set(local.columns))}")
if local.index.has_duplicates or fx.index.has_duplicates:
    raise ValueError("Duplicate dates")
if local.isna().any().any() or (local <= 0).any().any():
    raise ValueError("Local price panel contains missing or nonpositive prices")
if fx.reindex(local.index).isna().any():
    raise ValueError("Missing historical FX fixings on equity dates; audit calendars")
if (fx.reindex(local.index) <= 0).any():
    raise ValueError("Nonpositive EUR/NOK quotation")
print("Input date range:", local.index.min().date(), "to", local.index.max().date())
print("Tickers:", local.shape[1])

## 2. Transform and perform price-quality checks

Let \(S_t\) be the number of NOK per EUR, and \(P_t^{NOK}\) an Oslo-listed adjusted share price. Then \(P_t^{EUR}=P_t^{NOK}/S_t\). This is a currency conversion, **not** a hedging strategy.

In [ ]:
eur = local.copy()
for ticker in sorted(OSLO_NOK):
    eur[ticker] = local[ticker] / fx.reindex(local.index)
if eur.isna().any().any() or (eur <= 0).any().any():
    raise ValueError("Invalid converted price panel")
r = eur.pct_change(fill_method=None).dropna()
audit = pd.DataFrame({
    "Ticker": eur.columns,
    "Currency": ["NOK" if t in OSLO_NOK else "EUR (verify)" for t in eur.columns],
    "First_EUR": eur.iloc[0].values,
    "Last_EUR": eur.iloc[-1].values,
    "Largest_abs_daily_return_pct": (r.abs().max() * 100).reindex(eur.columns).values,
})
print(audit.sort_values("Largest_abs_daily_return_pct", ascending=False).head(12).round(3).to_string(index=False))
print("Warning: extreme adjusted returns require a corporate-action audit, not automatic deletion.")

## 3. Reconcile with the historical EUR panel

The existing validated panel, if available, is a useful accounting control. This comparison checks numeric agreement; it cannot prove that the original data source or corporate-action treatment is correct.

In [ ]:
REFERENCE = DATA / "prices_eur_2026-11-04.csv"
if REFERENCE.is_file():
    ref = pd.read_csv(REFERENCE, index_col=0, parse_dates=True).apply(pd.to_numeric, errors="coerce")
    ref.index = pd.to_datetime(ref.index).normalize()
    common_dates = ref.index.intersection(eur.index)
    common_tickers = ref.columns.intersection(eur.columns)
    if len(common_dates) and len(common_tickers):
        delta = (ref.loc[common_dates, common_tickers] - eur.loc[common_dates, common_tickers]).abs()
        print("Maximum absolute difference from previous panel:", float(delta.to_numpy().max()))
        print("Inspect differences before replacing the established source file.")
    else:
        print("Reference exists, but panels have no overlapping columns/dates")
else:
    print("No prior EUR panel in data/: comparison skipped")

## 4. Write audited research outputs

Outputs go to `reports/research_outputs/`; they are not official NextWise statements. Review file licensing and confidentiality before pushing generated data to GitHub.

In [ ]:
eur.to_csv(OUTPUT / "01_prices_eur_reconstructed.csv", index_label="Date")
audit.to_csv(OUTPUT / "01_currency_and_price_audit.csv", index=False)
print("Saved EUR adjusted panel and audit under", OUTPUT)

## Limitations and extensions

- An explicit source manifest with download timestamps and licensing is still needed.
- Trading-venue holidays and FX fixing calendars can disagree; do not silently forward-fill long gaps.
- Corporate actions (notably historical split/demerger events) require event-level review.
- In a future production pipeline, immutable raw data and versioned processing code should replace ad hoc overwrites.